# 🚀 Host Qwen 2.5 Coder 7B on Google Colab for OpenHands

This notebook turns your free Google Colab T4 GPU into a high-performance **OpenAI-compatible API endpoint** for **OpenHands**.

### Why this works:
- Uses **Ollama** with GPU acceleration (fast startup, low VRAM footprint).
- Uses **Qwen 2.5 Coder 7B** (state-of-the-art open coding model with native tool/function calling).
- Uses **Cloudflare Tunnel** (`trycloudflare.com`) to generate a free, secure public HTTPS endpoint with zero signup or API keys required.

### Step 1: Verify GPU
Ensure your Colab runtime is set to **T4 GPU** (`Runtime -> Change runtime type -> T4 GPU`).

In [ ]:
!nvidia-smi

### Step 2: Install Ollama & Cloudflare Tunnel

In [ ]:
# Install Ollama
!curl -fsSL https://ollama.com/install.sh | sh

# Install Cloudflared
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb > /dev/null 2>&1
print("✅ Ollama and Cloudflared installed successfully!")

### Step 3: Start Ollama and Pull Qwen 2.5 Coder 7B

In [ ]:
import subprocess, time

# Start Ollama daemon in the background
ollama_process = subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(3)

# Pull Qwen 2.5 Coder 7B (takes ~1-2 minutes)
print("📥 Downloading qwen2.5-coder:7b (optimized for coding & tool calling)...")
!ollama pull qwen2.5-coder:7b
print("✅ Model downloaded and ready on GPU!")

### Step 4: Expose Public OpenAI Endpoint via Cloudflare Tunnel
Run this cell to generate your public endpoint URL.

In [ ]:
import re, subprocess

# Launch Cloudflare Tunnel pointing to Ollama (port 11434)
tunnel_proc = subprocess.Popen(
    ["cloudflared", "tunnel", "--url", "http://localhost:11434"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True
)

public_url = None
print("⏳ Starting Cloudflare tunnel...")
for line in iter(tunnel_proc.stdout.readline, ""):
    match = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", line)
    if match:
        public_url = match.group(0)
        break

print("\n" + "="*60)
print("🎉 YOUR OPENAI-COMPATIBLE API IS LIVE!")
print("="*60)
print(f"Base URL: {public_url}/v1")
print(f"Model:    openai/qwen2.5-coder:7b")
print(f"API Key:  ollama (or any dummy text)")
print("="*60)
print("\n👉 Copy & paste these exact values into your OpenHands Settings:")
print(f"  • Model:    openai/qwen2.5-coder:7b")
print(f"  • Base URL: {public_url}/v1")
print(f"  • API Key:  ollama")
print("="*60)

# Keep the cell running to maintain the tunnel
try:
    tunnel_proc.wait()
except KeyboardInterrupt:
    print("\nTunnel stopped.")